# Platform Costs — collect, validate and publish

Independent monitoring workflow. Preview is the default. First run the BigQuery export and the additive monitoring DDL manually. Do not reset catalogs or rerun Azure pipelines. Publication requires explicit confirmation. The Job must have at most one concurrent run.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import sys

repository = next((root for root in (Path.cwd(), *Path.cwd().parents)
                   if (root / 'src/finops_cloud').is_dir() and (root / 'apps/finops_dashboard/platform_costs').is_dir()), None)
if repository is None:
    raise RuntimeError('Open this notebook from the cloned cloud Git folder; src and apps must be present.')
for source in (repository / 'src', repository / 'apps/finops_dashboard'):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))
    sys.path_importer_cache.pop(str(source), None)
importlib.invalidate_caches()
if importlib.util.find_spec('finops_cloud') is None:
    raise RuntimeError(f'Cannot import finops_cloud from {repository / "src"}. Check the Git files and restart Python, then Run All.')
print('Cloud repository paths resolved.')

In [ ]:
dbutils.widgets.dropdown('dry_run', 'true', ['true', 'false'])
dbutils.widgets.text('confirmation', '')
DRY_RUN = dbutils.widgets.get('dry_run') == 'true'
CONFIRMATION = dbutils.widgets.get('confirmation')
print('Preview only:', DRY_RUN)

In [ ]:
import importlib
import finops_cloud.monitoring.platform_costs as collector
importlib.reload(collector)

result = collector.run(spark, dbutils, confirmation=CONFIRMATION, dry_run=DRY_RUN)
display(spark.createDataFrame([result]))

## Expected result

`PREVIEW`: both sources validated, no table/object writes. Set `dry_run=false` and `confirmation=PUBLISH_PLATFORM_COSTS` only after checking scopes and publication approval.

`PUBLISHED`: latest aggregate snapshot replaced in `finops_ops.monitoring.platform_cost_monthly`, private audit written, and `gs://dtl_finops/platform_costs/published/latest.json` published. Run `04_validate_platform_costs.sql` in Databricks, then verify Cloud Run with GCS mode enabled. Partial periods remain partial even if the Job succeeds.